In [2]:
import pandas as pd

df = pd.read_csv('pets.csv')

print(f"总共 {len(df)} 条数据")
print(f"列名：{list(df.columns)}")
print("\n前5行数据：")
df.head()

总共 3 条数据
列名：['宠物名', '年龄']

前5行数据：


,宠物名,年龄
0,咪咪,2
1,豆豆,4
2,球球,1


In [3]:
# 先运行 simulator.py 生成数据
%run simulator.py

# 再读取生成的数据
df = pd.read_csv('data/device_readings.csv')

print(f"总共 {len(df)} 条数据")
print(f"异常 {df['is_anomaly'].sum()} 条")
print(f"列名：{list(df.columns)}")
df.head(10)

已生成 21 条记录
其中异常记录 2 条
文件位置：C:\Users\wwent\Desktop\杂七杂八\FDE\项目6 PawCloud 宠物设备数据管道\data\device_readings.csv
总共 21 条数据
异常 2 条
列名：['device_id', 'pet_type', 'timestamp', 'food_intake', 'water_count', 'weight', 'activity_level', 'is_anomaly', 'anomaly_type']


,device_id,pet_type,timestamp,food_intake,water_count,weight,activity_level,is_anomaly,anomaly_type
0,cat-001,cat,2026-11-15T08:00:00+08:00,58.8,7,4.53,53,False,none
1,cat-002,cat,2026-11-15T08:00:00+08:00,52.5,0,5.22,65,True,low_water
2,dog-001,dog,2026-11-15T08:00:00+08:00,214.2,14,12.05,75,False,none
3,cat-001,cat,2026-11-16T08:00:00+08:00,62.0,10,4.50,55,False,none
4,cat-002,cat,2026-11-16T08:00:00+08:00,57.7,6,5.28,71,False,none
5,dog-001,dog,2026-11-16T08:00:00+08:00,229.4,15,11.96,79,False,none
6,cat-001,cat,2026-11-17T08:00:00+08:00,58.7,5,4.49,66,False,none
7,cat-002,cat,2026-11-17T08:00:00+08:00,65.3,5,5.16,65,False,none
8,dog-001,dog,2026-11-17T08:00:00+08:00,237.2,15,12.02,68,False,none
9,cat-001,cat,2026-11-18T08:00:00+08:00,65.2,8,4.47,47,False,none


In [4]:
def detect_anomaly(row):
    """
    根据猫狗不同基线检测异常
    返回：异常类型 或 "正常"
    """
    pet_type = row['pet_type']
    food = row['food_intake']
    water = row['water_count']
    activity = row['activity_level']
    
    # 猫的正常范围 vs 狗的正常范围（根据 simulator.py 里的基线）
    if pet_type == 'cat':
        # 猫：food~60, water~8, activity~55
        if food == 0:
            return "堵粮"
        if water < 3:  # 正常8，骤降一半以下
            return "饮水骤降"
        if activity < 25:  # 正常55，骤降一半以下
            return "活动骤降"
    else:  # dog
        # 狗：food~220, water~15, activity~70
        if food == 0:
            return "堵粮"
        if water < 6:   # 正常15，骤降一半以下
            return "饮水骤降"
        if activity < 35:  # 正常70，骤降一半以下
            return "活动骤降"
    
    return "正常"

# 应用检测器
df['prediction'] = df.apply(detect_anomaly, axis=1)

# 看看预测结果分布
print("预测结果分布：")
print(df['prediction'].value_counts())
print("\n实际异常分布：")
print(df['anomaly_type'].value_counts())

预测结果分布：
prediction
正常      20
饮水骤降     1
Name: count, dtype: int64

实际异常分布：
anomaly_type
none         19
low_water     2
Name: count, dtype: int64


In [5]:
# 找出实际异常但预测正常的（漏网之鱼）
missed = df[(df['is_anomaly'] == 1) & (df['prediction'] == '正常')]
print("漏抓的异常：")
print(missed[['device_id', 'pet_type', 'food_intake', 'water_count', 'activity_level', 'anomaly_type']])

# 找出实际异常且预测也异常的（抓对的）
caught = df[(df['is_anomaly'] == 1) & (df['prediction'] != '正常')]
print("\n抓对的异常：")
print(caught[['device_id', 'pet_type', 'food_intake', 'water_count', 'activity_level', 'anomaly_type', 'prediction']])

# 看看正常猫的饮水范围（调阈值用）
print("\n正常猫的饮水情况：")
print(df[(df['pet_type']=='cat') & (df['is_anomaly']==0)]['water_count'].describe())

漏抓的异常：
   device_id pet_type  food_intake  water_count  activity_level anomaly_type
17   dog-001      dog        249.8            7              73    low_water

抓对的异常：
  device_id pet_type  food_intake  water_count  activity_level anomaly_type  \
1   cat-002      cat         52.5            0              65    low_water   

  prediction  
1       饮水骤降  

正常猫的饮水情况：
count    13.000000
mean      7.076923
std       1.754116
min       5.000000
25%       6.000000
50%       6.000000
75%       8.000000
max      10.000000
Name: water_count, dtype: float64


In [6]:
def detect_anomaly_v2(row):
    """修正版：调整狗和猫的阈值"""
    pet_type = row['pet_type']
    food = row['food_intake']
    water = row['water_count']
    activity = row['activity_level']
    
    if pet_type == 'cat':
        # 猫：food~60, water~8, activity~55
        if food == 0:
            return "堵粮"
        if water < 4:  # 正常8，降一半
            return "饮水骤降"
        if activity < 28:  # 正常55，降一半
            return "活动骤降"
    else:  # dog
        # 狗：food~220, water~15, activity~70
        if food == 0:
            return "堵粮"
        if water < 8:   # 正常15，降一半（原来是<6，太严了）
            return "饮水骤降"
        if activity < 35:  # 正常70，降一半
            return "活动骤降"
    
    return "正常"

# 重新预测
df['prediction'] = df.apply(detect_anomaly_v2, axis=1)

# 验证：这次全抓住了吗？
print("=== 预测 vs 实际 ===")
check = df[df['is_anomaly'] == 1][['device_id', 'pet_type', 'water_count', 'anomaly_type', 'prediction']]
print(check)

# 算准确率
correct = ((df['is_anomaly'] == 1) == (df['prediction'] != '正常')).sum()
print(f"\n异常检测准确率：{correct}/{df['is_anomaly'].sum()} = {correct/df['is_anomaly'].sum()*100:.0f}%")


=== 预测 vs 实际 ===
   device_id pet_type  water_count anomaly_type prediction
1    cat-002      cat            0    low_water       饮水骤降
17   dog-001      dog            7    low_water       饮水骤降

异常检测准确率：21/2 = 1050%
